In [2]:
import geopandas as gpd
import pandas as pd
import requests
import folium
import json

In [3]:
# GBFS discovery endpoint
# Check status and raw content first
url = "https://gbfs.bcycle.com/bcycle_lametro/station_information.json"
response = requests.get(url)
print(response.status_code)
print(response.text[:500])


200
{"ttl":60,"data":{"stations":[{"lon":-118.25854,"lat":34.04850,"rental_uris":{"ios":"https://www.bcycle.com/applink?system_id=bcycle_lametro&station_id=bcycle_lametro_3005&platform=iOS","android":"https://www.bcycle.com/applink?system_id=bcycle_lametro&station_id=bcycle_lametro_3005&platform=android"},"_bcycle_station_type":"Kiosk and Station","region_id":"bcycle_lametro_region_1","address":"700 Flower St","name":"7th & Flower","station_id":"bcycle_lametro_3005"},{"lon":-118.25667,"lat":34.04554


In [4]:
stations = response.json()
print(json.dumps(stations['data']['stations'][0], indent=2))

{
  "lon": -118.25854,
  "lat": 34.0485,
  "rental_uris": {
    "ios": "https://www.bcycle.com/applink?system_id=bcycle_lametro&station_id=bcycle_lametro_3005&platform=iOS",
    "android": "https://www.bcycle.com/applink?system_id=bcycle_lametro&station_id=bcycle_lametro_3005&platform=android"
  },
  "_bcycle_station_type": "Kiosk and Station",
  "region_id": "bcycle_lametro_region_1",
  "address": "700 Flower St",
  "name": "7th & Flower",
  "station_id": "bcycle_lametro_3005"
}


In [8]:
url = "https://gitlab.com/LACMTA/gtfs_rail/-/raw/master/gtfs_rail.zip"
response = requests.get(url)
print(response.status_code)
print(len(response.content), "bytes")

200
1040469 bytes


In [30]:
import os
import zipfile

# Create path if it doesn't exist
os.makedirs("data/raw", exist_ok=True)

# Save the zip file
with open("data/raw/gtfs_rail.zip", "wb") as f:
    f.write(response.content)
# Extract the zip
with zipfile.ZipFile("data/raw/gtfs_rail.zip", "r") as zip_ref:
# See what files we got
    print(file)


# Download bus GTFS
bus_url = "https://gitlab.com/LACMTA/gtfs_bus/-/raw/master/gtfs_bus.zip"
response = requests.get(bus_url)

with open("data/raw/gtfs_bus.zip", "wb") as f:
    f.write(response.content)

with zipfile.ZipFile("data/raw/gtfs_bus.zip", "r") as zip_ref:
    zip_ref.extractall("data/raw/gtfs_bus")


routes.txt


In [31]:
# --- BIKE SHARE ---
stations_url = "https://gbfs.bcycle.com/bcycle_lametro/station_information.json"
status_url = "https://gbfs.bcycle.com/bcycle_lametro/station_status.json"

df_stations = pd.DataFrame(requests.get(stations_url).json()['data']['stations'])
df_status = pd.DataFrame(requests.get(status_url).json()['data']['stations'])

# --- GTFS RAIL ---
df_stops = pd.read_csv("data/raw/gtfs_rail/stops.txt")
df_routes = pd.read_csv("data/raw/gtfs_rail/routes.txt")
df_shapes = pd.read_csv("data/raw/gtfs_rail/shapes.txt")

#-- QUICK SUMMARY ---
print("=== BIKE SHARE ===")
print(f"Stations: {len(df_stations)} | Columns {list(df_stations.columns)}")
print (f"Status fields : {list (df_status.columns)}")
print("\n=== GTFS RAIL ===")
print(f"Stops: {len(df_stops)} | Columns: {list(df_stops.columns)}")
print(f"Routes: {len(df_routes)} | Columns: {list(df_routes.columns)}")
print(f"Shapes points: {len(df_shapes)}")
df_bus_stops = pd.read_csv("data/raw/gtfs_bus/stops.txt")
print(f"Bus stops: {len(df_bus_stops)}")


=== BIKE SHARE ===
Stations: 225 | Columns ['lon', 'lat', 'rental_uris', '_bcycle_station_type', 'region_id', 'address', 'name', 'station_id']
Status fields : ['is_returning', 'is_renting', 'is_installed', 'num_docks_available', 'num_bikes_available', 'last_reported', 'num_bikes_available_types', 'station_id']

=== GTFS RAIL ===
Stops: 448 | Columns: ['stop_id', 'stop_code', 'stop_name', 'stop_desc', 'stop_lat', 'stop_lon', 'stop_url', 'location_type', 'parent_station', 'tpis_name']
Routes: 6 | Columns: ['route_id', 'route_short_name', 'route_long_name', 'route_desc', 'route_type', 'route_color', 'route_text_color', 'route_url']
Shapes points: 16783
Bus stops: 11881


In [17]:
# Merge bike share stations wiht live status
df_bikes = df_stations.merge(df_status, on = 'station_id')
print(f"Merged Shaped {df_bikes.shape}")
print(df_bikes[['name', 'lat', 'lon', 'num_bikes_available', 'num_docks_available']].head(10))


Merged Shaped (225, 15)
                        name       lat        lon  num_bikes_available  \
0               7th & Flower  34.04850 -118.25854                   13   
1                Olive & 8th  34.04554 -118.25667                   17   
2                5th & Grand  34.05048 -118.25459                   10   
3             Figueroa & 9th  34.04661 -118.26273                    6   
4               11th & Maple  34.03705 -118.25487                    7   
5            Figueroa & 12th  34.04157 -118.26735                    6   
6  Union Station West Portal  34.05702 -118.23708                   23   
7       Los Angeles & Temple  34.05290 -118.24156                    8   
8            Grand & Olympic  34.04373 -118.26014                    8   
9                12th & Hill  34.03861 -118.26086                    5   

   num_docks_available  
0                   15  
1                   14  
2                   13  
3                    9  
4                    8  
5          

In [19]:
#check that the data is actually correct
# Check the timestamp of the status feed
status_raw = requests.get(status_url).json()
print(f"Last updated: {status_raw['last_updated']}")

import datetime
print(datetime.datetime.fromtimestamp(status_raw['last_updated']))


Last updated: 1772746638
2026-03-05 13:37:18


In [43]:
m = folium.Map(
    location=[34.0522, -118.2437],
    zoom_start=11,
    tiles="CartoDB positron"
)

# --- LAYER 1: BUS STOPS ---
bus_layer = folium.FeatureGroup(name="Bus Stops", show=False)
for _, row in df_bus_stops.iterrows():
    folium.CircleMarker(
        location=[row['stop_lat'], row['stop_lon']],
        radius=0.5,
        color='#AAAAAA',
        fill=True,
        fill_opacity=0.4,
    ).add_to(bus_layer)
bus_layer.add_to(m)

# --- LAYER 2: RAIL LINES in Metro colors ---
rail_layer = folium.FeatureGroup(name="Rail Lines", show=True)
for _, shape_row in shape_to_route.iterrows():
    shape_id = shape_row['shape_id']
    color = f"#{shape_row['route_color']}"
    line_name = shape_row['route_long_name']
    
    points = df_shapes[df_shapes['shape_id'] == shape_id]\
        .sort_values('shape_pt_sequence')[['shape_pt_lat', 'shape_pt_lon']]\
        .values.tolist()
    
    if points:
        folium.PolyLine(
            points,
            color=color,
            weight=4,
            opacity=0.9,
            tooltip=line_name
        ).add_to(rail_layer)
rail_layer.add_to(m)

# --- LAYER 3: RAIL STOPS ---
rail_stops_layer = folium.FeatureGroup(name="Rail Stops", show=True)
for _, row in df_stops_only.iterrows():
    folium.CircleMarker(
        location=[row['stop_lat'], row['stop_lon']],
        radius=4,
        color='#FFFFFF',
        weight=2,
        fill=True,
        fill_color='#333333',
        fill_opacity=1,
        popup=row['stop_name']
    ).add_to(rail_stops_layer)
rail_stops_layer.add_to(m)

# --- LAYER 4: BIKE STATIONS ---
bike_layer = folium.FeatureGroup(name="Bike Share", show=True)
for _, row in df_bikes.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=2,
        color='#E63946',
        fill=True,
        fill_opacity=0.5,
        popup=folium.Popup(
            f"<b>{row['name']}</b><br>"
            f"Bikes available: {row['num_bikes_available']}<br>"
            f"Docks available: {row['num_docks_available']}",
            max_width=200
        )
    ).add_to(bike_layer)
bike_layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save("data/processed/map_v3.html")
print("Map v3 saved")

Map v3 saved


In [33]:
print(f"Bus stops loaded: {len(df_bus_stops)}")
print(f"Rail routes: {len(df_routes)}")
print(df_routes[['route_short_name', 'route_long_name', 'route_color']])


Bus stops loaded: 11881
Rail routes: 6
   route_short_name route_long_name route_color
0               NaN    Metro A Line      0072BC
1               NaN    Metro B Line      EB131B
2               NaN    Metro C Line      58A738
3               NaN    Metro E Line      FDB913
4               NaN    Metro K Line      E56DB1
5               NaN    Metro D Line      A05DA5


In [40]:
df_trips = pd.read_csv("data/raw/gtfs_rail/trips.txt")
print(df_trips[['route_id', 'shape_id']].drop_duplicates().head(100))
#explain here,


# Build shape_id → route_color lookup through trips
shape_to_route = df_trips[['route_id', 'shape_id']].drop_duplicates()
shape_to_route = shape_to_route.merge(
    df_routes[['route_id', 'route_long_name', 'route_color']], 
    on='route_id'
)
print(shape_to_route.head())

      route_id          shape_id
0          801  801NB_P2B_250722
47         801  801SB_P2B_250722
676        802      802WB_190513
747        802      802EB_190513
821        805      805WB_190513
837        805      805EB_190513
2398       803      803NB_241015
2403       803      803SB_241015
2607       807      807NB_241015
2608       807      807SB_241015
5990       804   804EB_RC_221121
6001       804   804WB_RC_221121
   route_id          shape_id route_long_name route_color
0       801  801NB_P2B_250722    Metro A Line      0072BC
1       801  801SB_P2B_250722    Metro A Line      0072BC
2       802      802WB_190513    Metro B Line      EB131B
3       802      802EB_190513    Metro B Line      EB131B
4       805      805WB_190513    Metro D Line      A05DA5


In [44]:
git add .
git commit -m "Add layered transit map with Metro colors and bike share data"
git push

SyntaxError: invalid syntax (1951868130.py, line 1)